## 0. Environment

In [ ]:
# Run once if needed
# !pip install -q torch --index-url https://download.pytorch.org/whl/cu121
# !pip install -q transformers==4.44.2 accelerate==0.34.2 peft==0.12.0
# !pip install -q datasets==2.21.0 sentence-transformers==3.0.1
# !pip install -q faiss-gpu-cu12   # or faiss-cpu
# !pip install -q bert-score==0.3.13 nltk tqdm pandas matplotlib

In [ ]:
import os, re, gc, json, time, string, random, zipfile, hashlib
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

# Paths — adjust if running outside Downloads
ROOT = Path.cwd()
LORA_ZIP = ROOT / 'qwen_hotpotqa_lora_final.zip'
LORA_DIR = ROOT / 'qwen_hotpotqa_lora_final'
ART = ROOT / 'enhanced_rag_artifacts'; ART.mkdir(exist_ok=True)
OUT = ROOT / 'enhanced_rag_outputs';   OUT.mkdir(exist_ok=True)

## 1. Load fine-tuned Qwen (LoRA) generator

Verbatim from Gomathi's email — float16, `device_map="auto"`, eval mode.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from peft import PeftModel

# Unzip the fine-tuned model files (idempotent)
if not LORA_DIR.exists():
    assert LORA_ZIP.exists(), f'Place {LORA_ZIP.name} next to this notebook.'
    with zipfile.ZipFile(LORA_ZIP, 'r') as z:
        z.extractall(LORA_DIR)

base_model_name = 'Qwen/Qwen3-4B-Instruct-2507'

qwen_tokenizer = AutoTokenizer.from_pretrained(str(LORA_DIR))
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map='auto',
)
qwen_ft_model = PeftModel.from_pretrained(base_model, str(LORA_DIR))
qwen_ft_model = qwen_ft_model.to(DEVICE)
qwen_ft_model.eval()
print('Fine-tuned Qwen model loaded successfully')

## 2. BGE-M3 embedder (matches Baseline `encode_texts` signature)

Loaded as raw HuggingFace `AutoModel` so the embedding call is identical to Gomathi's helper: `encode_texts(batch, tokenizer, model, device) -> np.ndarray`.

In [ ]:
EMBED_MODEL = 'BAAI/bge-m3'
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(EMBED_MODEL, torch_dtype=torch.float16).to(DEVICE)
embed_model.eval()
EMB_DIM = embed_model.config.hidden_size
print('BGE-M3 emb dim:', EMB_DIM)

@torch.inference_mode()
def encode_texts(texts: List[str], tokenizer=embed_tokenizer, model=embed_model,
                 device=DEVICE, max_length: int = 512) -> np.ndarray:
    """Mirrors the helper used in Gomathi's Baseline RAG notebook.
    BGE-M3 dense retrieval = L2-normalized [CLS] from the last hidden state."""
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
    out = model(**enc)
    cls = out.last_hidden_state[:, 0]
    cls = torch.nn.functional.normalize(cls, p=2, dim=1)
    return cls.float().cpu().numpy()

## 3. Build `full_train_chunk_records` (fast — ~minutes)

Same chunking convention as the Baseline notebook: walk every train example's `context`, dedup by `(title, sent_id)`, store sentence-level passages with provenance.

In [ ]:
from datasets import load_dataset
ds = load_dataset('hotpotqa/hotpot_qa', 'distractor', trust_remote_code=True)
print(ds)

In [ ]:
RECORDS_PATH = ART / 'full_train_chunk_records.jsonl'

def build_full_train_records():
    seen = set(); records = []
    for ex in tqdm(ds['train'], desc='Chunking train'):
        for title, sents in zip(ex['context']['title'], ex['context']['sentences']):
            for sid, s in enumerate(sents):
                t = s.strip()
                if len(t) < 3:
                    continue
                key = (title, sid, hashlib.md5(t.encode('utf-8')).hexdigest()[:8])
                if key in seen:
                    continue
                seen.add(key)
                records.append({
                    'chunk_id': len(records),
                    'title': title,
                    'sent_id': sid,
                    'text': t,
                    'passage': f'{title}. {t}',
                })
    return records

if RECORDS_PATH.exists():
    full_train_chunk_records = [json.loads(l) for l in open(RECORDS_PATH, 'r', encoding='utf-8')]
    print(f'Loaded {len(full_train_chunk_records):,} cached chunks from {RECORDS_PATH}')
else:
    full_train_chunk_records = build_full_train_records()
    with open(RECORDS_PATH, 'w', encoding='utf-8') as f:
        for r in full_train_chunk_records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'Wrote {len(full_train_chunk_records):,} chunks to {RECORDS_PATH}')

full_chunk_texts = [r['passage'] for r in full_train_chunk_records]
print('First chunk:', full_train_chunk_records[0])

## 4. Build `full_embeddings` and `index_full` (~3 h, one-time)

Cell mirrors the screenshot from Gomathi's Baseline notebook: `batch_size=64`, `encode_texts(batch_chunks, tokenizer, model, device)`, append → `np.vstack`, then a flat-IP FAISS index.

In [ ]:
EMB_PATH   = ART / 'full_embeddings.npy'
INDEX_PATH = ART / 'index_full.faiss'

if EMB_PATH.exists():
    full_embeddings = np.load(EMB_PATH)
    print('Loaded cached embeddings:', full_embeddings.shape)
else:
    full_embeddings = []
    batch_size = 64
    for start_idx in tqdm(range(0, len(full_chunk_texts), batch_size)):
        batch_chunks = full_chunk_texts[start_idx:start_idx + batch_size]
        batch_embeddings = encode_texts(
            batch_chunks,
            qwen_tokenizer if False else embed_tokenizer,  # always BGE-M3 tokenizer
            embed_model,
            DEVICE,
        )
        full_embeddings.append(batch_embeddings)
    full_embeddings = np.vstack(full_embeddings).astype('float32')
    print('Full embedding shape:', full_embeddings.shape)
    np.save(EMB_PATH, full_embeddings)

In [ ]:
import faiss

if INDEX_PATH.exists():
    index_full = faiss.read_index(str(INDEX_PATH))
    print('Loaded cached FAISS index, ntotal=', index_full.ntotal)
else:
    index_full = faiss.IndexFlatIP(full_embeddings.shape[1])
    index_full.add(full_embeddings)
    faiss.write_index(index_full, str(INDEX_PATH))
    print('Built FAISS index, ntotal=', index_full.ntotal)

## 5. Cross-encoder reranker (Enhanced-RAG specific)

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
RERANK_MODEL = 'BAAI/bge-reranker-v2-m3'
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE, max_length=512)
print('Loaded reranker:', RERANK_MODEL)

## 6. Generation helper (uses fine-tuned Qwen)

In [ ]:
@torch.inference_mode()
def llm_generate(messages: List[Dict], max_new_tokens: int = 64, temperature: float = 0.0) -> str:
    prompt = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(prompt, return_tensors='pt').to(qwen_ft_model.device)
    out = qwen_ft_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=0.9,
        pad_token_id=qwen_tokenizer.pad_token_id,
    )
    text = qwen_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()

## 7. Enhanced RAG components

In [ ]:
REWRITE_SYS = (
    'You rewrite multi-hop questions for a retrieval system. '
    'Given a question, write ONE short hypothetical sentence that, if true, would answer it. '
    'Do not hedge. Output only the sentence.'
)

def rewrite_query(q: str) -> str:
    msgs = [
        {'role': 'system', 'content': REWRITE_SYS},
        {'role': 'user', 'content': f'Question: {q}\nHypothetical answer sentence:'},
    ]
    return llm_generate(msgs, max_new_tokens=48, temperature=0.0)

def faiss_topn(question: str, hyde: Optional[str], n: int = 30) -> List[int]:
    queries = [question] + ([hyde] if hyde else [])
    qv = encode_texts(queries)
    qv = qv.mean(axis=0, keepdims=True)
    qv = qv / (np.linalg.norm(qv, axis=1, keepdims=True) + 1e-9)
    _, I = index_full.search(qv.astype('float32'), n)
    return I[0].tolist()

def rerank(question: str, cand_idx: List[int], k: int) -> Tuple[List[int], List[float]]:
    if not cand_idx:
        return [], []
    pairs = [(question, full_train_chunk_records[i]['passage']) for i in cand_idx]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))[:k]
    return [cand_idx[i] for i in order], [float(scores[i]) for i in order]

def filter_context(top_idx: List[int], top_scores: List[float], min_score: float) -> List[Dict]:
    kept = [full_train_chunk_records[i] for i, s in zip(top_idx, top_scores) if s >= min_score]
    if not kept and top_idx:
        kept = [full_train_chunk_records[top_idx[0]]]
    return kept

def format_context(chunks: List[Dict], max_chars: int = 3500) -> str:
    out, total = [], 0
    for c in chunks:
        line = f"[{c['title']}] {c['text']}"
        if total + len(line) > max_chars:
            break
        out.append(line); total += len(line)
    return '\n'.join(out)

QA_SYS = (
    'You answer multi-hop questions using ONLY the provided context. '
    'Answer in 1-5 words. If the context is insufficient, give your best guess from the context. '
    'Do not explain. Do not repeat the question.'
)

def generate_answer(question: str, ctx_chunks: List[Dict], temperature: float = 0.0,
                    max_new_tokens: int = 32) -> str:
    msgs = [
        {'role': 'system', 'content': QA_SYS},
        {'role': 'user', 'content': f'Context:\n{format_context(ctx_chunks)}\n\nQuestion: {question}\nAnswer:'},
    ]
    return llm_generate(msgs, max_new_tokens=max_new_tokens, temperature=temperature)

## 8. End-to-end Enhanced RAG

In [ ]:
def enhanced_rag(question: str, n_first: int = 30, k_rerank: int = 5,
                 use_hyde: bool = True, filter_min_score: float = float('-inf'),
                 temperature: float = 0.0) -> Dict:
    hyde = rewrite_query(question) if use_hyde else None
    cand = faiss_topn(question, hyde, n=n_first)
    top_idx, top_scores = rerank(question, cand, k=k_rerank)
    ctx = filter_context(top_idx, top_scores, min_score=filter_min_score)
    pred = generate_answer(question, ctx, temperature=temperature)
    return {'pred': pred, 'retrieved': ctx, 'hyde': hyde,
            'top_idx': top_idx, 'top_scores': top_scores}

# Smoke test
demo_q = ds['validation'][0]['question']
demo_a = ds['validation'][0]['answer']
demo = enhanced_rag(demo_q)
print('Q   :', demo_q)
print('Gold:', demo_a)
print('HyDE:', demo['hyde'])
print('Pred:', demo['pred'])
for c, s in zip(demo['retrieved'][:3], demo['top_scores'][:3]):
    print(f"  [{c['title']}#{c['sent_id']}] (rerank={s:+.2f}) {c['text'][:100]}")

## 9. Evaluation harness (matches `arag/src/arag/evaluation/metrics.py`)

In [ ]:
def normalize(s: str) -> str:
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def em_score(pred: str, gold: str) -> float:
    return float(normalize(pred) == normalize(gold))

def f1_score(pred: str, gold: str) -> float:
    pt, gt = normalize(pred).split(), normalize(gold).split()
    if not pt or not gt:
        return float(pt == gt)
    common = Counter(pt) & Counter(gt)
    n = sum(common.values())
    if n == 0:
        return 0.0
    p, r = n / len(pt), n / len(gt)
    return 2 * p * r / (p + r)

def support_recall(retrieved: List[Dict], example) -> float:
    gold = set(zip(example['supporting_facts']['title'], example['supporting_facts']['sent_id']))
    if not gold:
        return float('nan')
    got = set((c['title'], c['sent_id']) for c in retrieved)
    return len(gold & got) / len(gold)

## 10. Run on n=50 dev (matches Gomathi's reported sample size for direct comparability)

In [ ]:
N_EVAL = 50
eval_idx = random.sample(range(len(ds['validation'])), N_EVAL)
dev_eval = ds['validation'].select(eval_idx)

rows = []
for ex in tqdm(dev_eval, desc='Enhanced RAG'):
    out = enhanced_rag(ex['question'], n_first=30, k_rerank=5, use_hyde=True,
                       filter_min_score=float('-inf'), temperature=0.0)
    rows.append({
        'id': ex['id'], 'question': ex['question'], 'gold': ex['answer'], 'pred': out['pred'],
        'type': ex['type'], 'level': ex['level'],
        'em': em_score(out['pred'], ex['answer']),
        'f1': f1_score(out['pred'], ex['answer']),
        'support_recall': support_recall(out['retrieved'], ex),
    })
df_enh = pd.DataFrame(rows)
df_enh.to_csv(OUT / 'predictions_enhanced.csv', index=False)
print(df_enh[['em','f1','support_recall']].mean().round(3))

In [ ]:
# BERTScore (matches §V.B of the project plan)
from bert_score import score as bertscore_fn
P, R, F = bertscore_fn(df_enh['pred'].tolist(), df_enh['gold'].tolist(),
                       model_type='microsoft/deberta-xlarge-mnli', lang='en',
                       verbose=False, batch_size=32, device=DEVICE)
bertscore_f1 = float(F.mean())
print(f'BERTScore-F1: {bertscore_f1:.3f}')

## 11. Validation-based hyperparameter tuning

Per Gomathi's email — Baseline values are a starting point but Enhanced has its own axes (`n_first`, `k_rerank`, `use_hyde`, `filter_min_score`). Running on a held-out 30-sample slice.

In [ ]:
from itertools import product

TUNE_N = 30
tune_idx = [i for i in random.sample(range(len(ds['validation'])), TUNE_N + N_EVAL) if i not in set(eval_idx)][:TUNE_N]
dev_tune = ds['validation'].select(tune_idx)

grid = list(product(
    [20, 30],          # n_first
    [3, 5, 8],         # k_rerank
    [True, False],     # use_hyde
    [float('-inf'), 0.0],  # filter_min_score
))

tune_rows = []
for n_first, k_r, hyde, mn in tqdm(grid, desc='Tuning'):
    if k_r > n_first: continue
    em_v, f1_v = [], []
    for ex in dev_tune:
        out = enhanced_rag(ex['question'], n_first=n_first, k_rerank=k_r,
                           use_hyde=hyde, filter_min_score=mn, temperature=0.0)
        em_v.append(em_score(out['pred'], ex['answer']))
        f1_v.append(f1_score(out['pred'], ex['answer']))
    tune_rows.append({'n_first': n_first, 'k_rerank': k_r, 'use_hyde': hyde,
                      'min_score': mn, 'EM': float(np.mean(em_v)), 'F1': float(np.mean(f1_v))})
tune_df = pd.DataFrame(tune_rows).sort_values('F1', ascending=False)
tune_df.to_csv(OUT / 'sweep_enhanced.csv', index=False)
print(tune_df.head(8).to_string(index=False))

## 12. Comparison table + bar chart

Combines this notebook's Enhanced-RAG result with the team-reported Baseline / Agentic figures (Apr 25 email). Re-run those rows from the team repo if you want fresh numbers under the same RNG.

In [ ]:
summary = pd.DataFrame([
    {'system': 'Baseline RAG (Gomathi, n=50)',         'EM': 0.260, 'F1': 0.315, 'BERTScore-F1': np.nan},
    {'system': 'Agentic Hierarchical RAG (Chuan, n=50)','EM': 0.340, 'F1': 0.430, 'BERTScore-F1': np.nan},
    {'system': 'Enhanced RAG (this notebook, n=50)',    'EM': float(df_enh['em'].mean()),
                                                        'F1': float(df_enh['f1'].mean()),
                                                        'BERTScore-F1': bertscore_f1},
])
summary.to_csv(OUT / 'summary_comparison.csv', index=False)
print(summary.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(summary)); w = 0.28
ax.bar(x - w, summary['EM'],  w, label='EM')
ax.bar(x,     summary['F1'],  w, label='F1')
ax.bar(x + w, summary['BERTScore-F1'].fillna(0), w, label='BERTScore-F1')
ax.set_xticks(x)
ax.set_xticklabels(summary['system'], rotation=15, ha='right')
ax.set_ylabel('score'); ax.set_ylim(0, 1)
ax.set_title('HotpotQA dev (n=50) — fine-tuned Qwen3-4B-Instruct + LoRA')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / 'comparison_bar.png', dpi=150); plt.show()

In [ ]:
# Per-type / per-level breakdowns (addresses Prof. Barman's Attempt-3 feedback on experimental setup)
print('Enhanced RAG by question type:')
print(df_enh.groupby('type')[['em','f1','support_recall']].mean().round(3))
print('\nEnhanced RAG by difficulty:')
print(df_enh.groupby('level')[['em','f1','support_recall']].mean().round(3))